# 04 · hypathia

**Purpose.** Peaks (PMaps) found in the detsim waveforms.

**Inputs.** `<sample>_<i>.hypathia.h5` (`/PMAPS/S1`, `/PMAPS/S2`, per-sensor tables).

**Outputs.** None.

**What is checked.** number of S1 and S2 peaks per event, S1 and S2 energies, fibers vs PMTs on the same events, events lost (empty PMaps).

**Next step.** `05_dorothea_kr` (Kr) or `06_qbb` (0νββ). Run with `./submit.sh hypathia <fibers|pmts> <sample>`.

In [ ]:
# Setup: run `source env.sh` before starting Jupyter, or pick the IC-3.8-2024-06-08 kernel.
import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks, fits
print("ICDIR =", paths.ICDIR or "NOT SET (source env.sh)")

In [ ]:
EL_GAIN = paths.EL_GAIN   # gain of the new runs to read (folder <det>_g<EL_GAIN>)
SAMPLE, I, USE_NEW = "20250624_Next100_1Kev_ELoff_Kr", 1, False
def hyp_file(det):
    fs = paths.new_files(SAMPLE, det, "hypathia", EL_GAIN) if USE_NEW else paths.old_files(SAMPLE, det, "hypathia")
    return [f for f in fs if paths.file_index(f.replace("_PMTs", "")) == I][0]
P = {}
for det in ["fibers", "pmts"]:
    f = hyp_file(det); print(det, f)
    s1 = pd.read_hdf(f, "/PMAPS/S1"); s2 = pd.read_hdf(f, "/PMAPS/S2")
    ev = pd.read_hdf(f, "/Run/events")
    P[det] = dict(n_events=len(ev),
                  s1=s1.groupby(["event", "peak"]).ene.sum().groupby("event").agg(["count", "sum"]),
                  s2=s2.groupby(["event", "peak"]).ene.sum().groupby("event").agg(["count", "sum"]))
for det, p in P.items():
    print(f"{det:6s}: {p['n_events']} events with PMaps; events with 1 S1: {(p['s1']['count']==1).sum()}, with 1 S2: {(p['s2']['count']==1).sum()}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for det, p in P.items():
    ax[0].hist(p["s1"]["sum"], np.linspace(0, 60, 61), histtype="step", lw=2, label=det)
    ax[1].hist(p["s2"]["sum"], np.linspace(0, 1.2e4, 61), histtype="step", lw=2, label=det)
ax[0].set_xlabel("S1 energy per event (pes)"); ax[1].set_xlabel("S2 energy per event (pes)")
for a in ax: a.legend()
r1 = P["fibers"]["s1"]["sum"].median() / P["pmts"]["s1"]["sum"].median()
r2 = P["fibers"]["s2"]["sum"].median() / P["pmts"]["s2"]["sum"].median()
print(f"median ratio fibers/PMTs: S1 {r1:.2f} (placeholder, see 03), S2 {r2:.2f}")